# Dealing With Overlapping Subproblems

In previous lectures, we used divide and conquer to solve problems by splitting them into subproblems, solving each recursively, and combining the results. In algorithms like merge sort and binary search, the subproblems do not overlap — they are solved independently of each other.

Today we look at problems where the subproblems **do** overlap. This leads to a powerful technique called **dynamic programming**.

An example of non-overlapping subproblems:
```python
def merge_sort(a_list):
    left = a_list[0 : len(a_list)//2]
    right = a_list[len(a_list)//2 : len(a_list)]
    
    sorted_left = merge_sort(left)
    sorted_right = merge_sort(right)   
    
    return merge(sorted_left, sorted_right)
```

The sorting of the left sublist is independent of the sorting of the right sublist. They do not share any work.

---

## Overlapping Subproblems

Consider this problem: compute the number of distinct ways to climb a staircase with `n` steps, where at each step you may either climb 1 or 2 steps.

```python
def climb(n):
    if n == 0 or n == 1:
        return 1
    return climb(n - 1) + climb(n - 2)
```

Sidenote: many recursive designs are conceptualized/strategized by analyzing **the first choice**

This is slightly different from divide and conquer. Using this technique, our strategy involves a sequence of choices.  And if we know how to many the first choice correctly, that solves most of the problem.

In [ ]:
def climb(n):
    if n == 0 or n == 1:
        return 1
    return climb(n - 1) + climb(n - 2)

In [ ]:
climb(40)

why is this slow?

- climb(40) calls (depends on) climb(39) and climb(38)
- climb(39) calls (depends on) climb(38) and climb(37)


`climb(38)` is called twice.  This is inefficient. We'll fix this. But first, what is T(n)?

$$T(n) = T(n-1) + T(n-2)$$

Typically, we want to know how fast an algorithm is. So, we use $O$. This allows us to say something like: The running time is $O(n^2)$ or **at most** $n^2$.  This is an upperbound estimate.

Here's the analysis is different. We know that `climb` is slow. So question is: how slow is this? To answer this, we do a **lower bound** estimate. We want to be able to say something like. The running time is **at least** such and such...  So, we have to use $\Omega$ to estimate.

$$T(n) \ge T(n-2) + T(n-2) = 2*T(n-2)$$

In fact:
$$2*T(n-1) \ge T(n) \ge 2*T(n-2)$$

$$2^2*T(n-2) \ge T(n) \ge 2^2 *T(n-4)$$

We have established this: $T(n) \ge 2 * T(n-2)$

replace `n` with `apple`: $T(apple) \ge 2 * T(apple - 2)$

replace `n` with `n-2`: $T(n-2) \ge 2 * T(n-4)$

substitute into original: $T(n) \ge 2 * T(n-2) \ge 2 * 2 * T(n-4)$

$$2^3*T(n-3) \ge T(n) \ge 2^3 *T(n-6)$$

Original equation: T(n) >= 2 * T(n-2)

Replace n with n-6: T(n-6) >= 2*T(n-8)

$$2^4*T(n-4) \ge T(n) \ge 2^4 *T(n-8)$$

After $k$ steps:

$$2^k*T(n-k) \ge T(n) \ge 2^k *T(n-2*k)$$


$$2^n * T(0) \ge T(n) \ge 2^{n \over 2} * T(0)$$

$$2^n \ge T(n) \ge (\sqrt{2})^n$$

$$2^n \ge T(n) \ge 1.41^n$$

So $T(n)$ is exponential. It should be something like $T(n) = \phi^n$, where $\sqrt{2} < \phi < 2$

Original equation: $T(n) = T(n-1) + T(n-2)$.  Because we know that $T(n) = \phi^n$.

We can say that $\phi^n = \phi^{n-1} + \phi^{n-2}$

This means: $\phi^2 = \phi + 1$

So we have: $\phi^2 - \phi - 1  = 0$

$\phi = {1 \pm \sqrt{1 + 4} \over 2} = {1 \pm \sqrt{5} \over 2}$

Cannot be negative; only solution is $\phi = {1 + \sqrt{5} \over 2}$

In [ ]:
import math
(1 + math.sqrt(5))/2

$$T(n) \approx 1.62^n$$

In [ ]:
1.62**50

How do we make this faster? don't calulate anything (recursive call) twice. Calculate once, store it, and return it if you need it again.

Why is it slow? it calculates many recursive calls more than once.

Use `Cache[k]` to store the value of `climb(k)`.  This is a dictionary: key is `k` (input), the stored value is `Cache[k]` (output)


In [ ]:
Cache = {}
def climb(n):
    if n in Cache:
        return Cache[n]
        
    if n == 0 or n == 1:
        Cache[n] = 1
        return 1
    output = climb(n - 1) + climb(n - 2)
    Cache[n] = output
    return output

In [ ]:
climb(2000)

---

### Analyzing the running time of `climb`

Let's analyze the running time equation of `climb(n)`.

$T(n) = T(n-1) + T(n-2)$

$T(n) \ge T(n-2) + T(n-2) = 2 \cdot T(n-2)$

$T(n) \ge 2 \cdot T(n-2)$

$T(n) \ge 2^2 \cdot T(n-4) \ge 2^3 \cdot T(n-6)$

$T(n) \ge 2^4 \cdot T(n-8)$

After $n/2$ steps: $T(n) \ge 2^{n/2} \cdot T(n - n)$

$T(n) \ge 2^{n \over 2}$

This is **exponential**. The redundant calculations make `climb` extremely slow for large inputs.

In [ ]:
n = 50
2**(n/2) / 2**(43/2)

In overlapping subproblems, subproblems are calculated more than once.

---

### Speeding things up: store and look up

We will store and look up stored values to speed things up.

1. Which calculations do we store?
    - the output of climb. When we see a return, that's where the output is.

2. When do we store them?
    - just before an output is returned

3. Where do we store them?
    - some table, a dictionary

4. How do we store them?
    - `table[input] = output = climb(input)`
    - this way, `table[input]` stores the output of `climb(input)`

5. When do we look them up?
    - first thing, before we do any calculation.

In [ ]:
Table = {}

def climb(n):
    # look up to see if climb(n) has been calculated already?
    if n in Table:
        return Table[n]
    
    if n == 0 or n == 1:
        output = 1
        Table[n] = output
        return output
    output = climb(n - 1) + climb(n - 2)
    Table[n] = output
    return output

print(climb(10))
print(Table)

In [ ]:
climb(200)

---

### Making Change

Determine if it is possible to make change for an amount `A` using some coin values: `c_1`, ..., `c_n`.

Examples:
+ A=13, coin values=[5, 6, 3].   Output: True.  13 = 5 + 5 + 3
+ A=11, coin values=[5, 6, 3].   Output: True.  11 = 5 + 6
+ A=8, coin values=[5, 6, 3].   Output: True. 8 = 5 + 3
+ A=7, coin values=[5, 6, 3].   Output: False

Problem (English version): "`Is it possible to make change for 123 using [5, 6, 3]?`"

Problem (Python version): "`is_changeable(123, [5,6,3])`"


There are 3 possibilities we can make for the first move:

(1) Use 6 
- New Problem: "`Is it possible to make change 117 using [5, 6, 3]?`"
- Python: "`is_changeable(117, [5,6,3])`"
  
(2) Use 5 
- New Problem: "`Is it possible to make change 118 using [5, 6, 3]?`"
- Python: "`is_changeable(118, [5,6,3])`"
  
(3) Use 3 
- New Problem: "`Is it possible to make change 120 using [5, 6, 3]?`"
- Python: "`is_changeable(120, [5,6,3])`"


English: `It's possible to make change for 123 if it's possible to make change for either 117, or 118, or 120.`


#PID:46


English version: `It's possible to make change for 123 if it's possible to make change for either 117, or 118, or 120.`

Python version: `is_changeable(127,[5,6,3]) = is_changeable(117,[5,6,3]) or is_changeable(118,[5,6,3]) or is_changeable(120,[5,6,3])`


English: `It's possible to make change for A using [c1,...,cN] if it's possible to make change for A-c1, or ... or A-cN`

Python: `is_changeable(A, [c1,...,cN]) = is_changeable(A-c1,[c1,...,cN]) or ... or is_changeable(A-cN, [c1,...,cN])`




### Declarative: WHAT

English: `It's possible to make change for A using "coins", if it's possible to make change for A-c for any coin c in "coins"`


Python: `is_changeable(A, coins) = any(is_changeable(A-c, coins) for c in coins)`

### Procedural: HOW

English: To find if we can make change for A using coins, we go through each coin c, and check if we can make change for A-c. If we can make change for A-c, for any coin c, then yes we can make change for A.  If we cannot make change for A-c for any value c, then we cannot make change for A.

Python:
```python
def is_changeable(A, coins):
    for c in coins:
        if is_changeable(A-c, coins):
            return True
    return False  # Note: if you don't get the indentation correctly, your coding skill is weak
```

In [ ]:
any([False, False, False])

In [ ]:
#PID:47

def is_changeable(A, coins):
    if A==0:
        return True
    if A<0:
        return False
    return any(is_changeable(A-c, coins) for c in coins)

def is_changeable(A, coins):
    if A==0:
        return True
    if A<0:
        return False
    for c in coins:
        if is_changeable(A-c, coins):
            return True
    return False 


In [ ]:

is_changeable(100, [1,2])

#PID:48

Use a table to cache the values of this function so that you do not calculate a value twice

```python
def is_changeable(A, coins):
    if A==0:
        return True
    if A<0:
        return False
    for c in coins:
        if is_changeable(A-c, coins):
            return True
    return False 
```
    

#### Caching -- storing outputs so you don't to calculate something more than once

Store the "output" in a cache, using the "input" as the key.  

$$Cache[ input ] \leftarrow output$$

This way we can check if the function is already computed for an input.

In [ ]:
Cache = {}
def is_changeable(A, coins):
    if A in Cache:
        return Cache[A]
        
    if A==0:
        Cache[A] = True
        return True
    if A<0:
        Cache[A] = False
        return False
    for c in coins:
        if is_changeable(A-c, coins):
            Cache[A] = True
            return True

    Cache[A] = False
    return False 

In [ ]:
is_changeable(15, [5,11])

In [ ]:
is_changeable(27, [5,11])

In [ ]:
is_changeable(37, [5,11])

In [ ]:
is_changeable(16, [5,11])

In [ ]:
is_changeable(13, [5,11])

In [ ]:
is_changeable(16, [5,13])

In [ ]:
Cache

In [ ]:
#
#  Task: make this faster
#

Table = {}
def something(m,n):
    if m<0:
        return 5
    if n<0:
        return 10
    
    if m==n:
        return something(m-1, n-1)
    else:
        return something(m-1, n) + something(m, n-1)


In [ ]:
something(4,5)

---

### Counting Paths

Write a function `count_paths(m, n)` that counts the number of paths to move from the top-right (m,n) (start) to bottom-left (1,1) (finish) on an m×n grid, moving only left or down.

Example (2×2 grid):
```
(2,2) → (1,1)
Path 1: (2,2) → (1,2) → (1,1)
Path 2: (2,2) → (2,1) → (1,1)
count_paths(2,2) = 2
```

In this figure below, we start from (7,3), and try to get to (1,1).
<img src="https://i.imgur.com/75jsBWm.png">

start = (3,3)

How many different paths (ways) to move from (3,3) to (1,1).


In [ ]:
#PID:49
#
# Solve the simplest cases
#
def count_paths(x,y):
    if x<=1 or y<=1:
        return 1

    

In [ ]:
#PID:50
#
# Input: source x, y
# Output: number of ways to get from (x,y) to (1,1)
# API: count_path(x,y) = the number of ways to get from (x,y) to (1,1)
#
def count_paths(x,y):
    if x<=1 or y<=1:
        return 1
    return count_paths(x-1,y) + count_paths(x,y-1)

In [ ]:
count_paths(11,11)

#PID:51

Why is this `count_paths(20,20)` so slow?

`count_paths(19,19)` is calculated twice.

`count_paths(18,18)` is calculated 4 times.

etc.

#PID:52

$T(m, n)$ -- r.t. equation , $m>n$.

Show that: $T(m,n) \ge T(n,n)$.  WHy? because $m>n$.

#PID:53


Show that: $T(n,n) \ge 2*T(n-1,n-1)$

T(n,n) = T(n-1,n) + T(n,n-1) > T(n-1) + T(n-1)

T(n-1,n) > T(n-1,n-1)

T(n,n-1) > T(n-1,n-1)

$$T(n-1,n-1) \ge 2 * T(n-2,n-2)$$

$T(m,n) > T(n,n) > 2*T(n-1,n-1) > 2^2*T(n-2,n-2)$


$T(m,n) > T(n,n) > 2*T(n-1,n-1) > 2^2*T(n-2,n-2) > 2^3 * T(n-3,n-3)$

$T(m,n) > 2^{n-1} * T(1,1)$

In [ ]:
count_paths(5,5)

In [ ]:
#PID:54

# Make this quick
Cache = {}
# key = (x,y)
# value = output of count_paths(x,y)
# Cache[key] = value

def count_paths(x,y):
    # check if this calculation has been computed
    if (x,y) in Cache:
        return Cache[x,y]
    
    if x<=1 or y<=1:
        Cache[x,y] = 1
        return 1
    Cache[x,y] = count_paths(x-1,y) + count_paths(x,y-1)
    return Cache[x,y]
    

In [48]:
count_paths(200,200)

25802631612885822800244581533935335026869906110545776499962170319780283802669663809106916170169547105655150024437788000


Amortized analysis: with caching, running time is $O(n^2)$

Non-caching, running time is $\Omega(2^n)$

In [58]:
n = 50
(2**n / (n*n)) / 1e9

450.3599627370496